In [ ]:
from pathlib import Path
import av
import numpy as np
import torch
import polars as pl

# Config
video_dir = Path("../data/video")
csv_path = "../data/view_counts.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
df = pl.read_csv(csv_path)
df = df.filter(pl.col("url").str.contains("watch"))
print(f"{df.shape[0]} valid videos")

In [ ]:
from transformers import XCLIPProcessor, XCLIPModel, AutoTokenizer, AutoModel

# Load model and processor
processor = XCLIPProcessor.from_pretrained("microsoft/xclip-base-patch32")
model = XCLIPModel.from_pretrained("microsoft/xclip-base-patch32").to(device)
model.eval()

# Load tokenizer and model
text_tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
text_model = AutoModel.from_pretrained("Intel/dynamic_tinybert").to(device)
text_model.eval()

In [ ]:
# Get unique channel list
channels = df.select("channel").unique().to_series()

# List to collect per-channel normalized DataFrames
normalized_chunks = []

# Loop through channels and normalize
for ch in channels:
    channel_df = df.filter(pl.col("channel") == ch)

    # log scale view_count
    channel_df = channel_df.with_columns(
        pl.col("view_count").log().alias("log_view_count")
    )

    # set target to log view_count
    channel_df = channel_df.with_columns(
        pl.col("log_view_count").alias("target")
    )
    
    # Normalize view_count to [0, 1]
    min_vc = channel_df["target"].min()
    max_vc = channel_df["target"].max()
    channel_df = channel_df.with_columns(
        ((pl.col("target") - min_vc) / (max_vc - min_vc)).alias("target")
    )

    # Add min and max for debugging
    channel_df = channel_df.with_columns(
        pl.lit(min_vc).alias("min_log_view_count"),
        pl.lit(max_vc).alias("max_log_view_count")
    )

    normalized_chunks.append(channel_df)

# Concatenate all normalized chunks
norm_df = pl.concat(normalized_chunks)

norm_df.shape

In [ ]:
import torch
import numpy as np
import av
from tqdm.notebook import tqdm


# --- Config ---
num_sampled_frames = 8

# --- Util Functions ---
def get_video_path(url):
    return video_dir / f"{url.split('watch?v=')[-1]}_1080p24.mp4"

def read_pyav(container, indices):
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i > indices[-1]:
            break
        if i in indices:
            frame = frame.to_rgb().to_ndarray()
            frames.append(frame)
    return frames

def sample_frames(container, num_frames):
    total = container.streams.video[0].frames
    indices = np.linspace(0, total - 1, num=num_frames).astype(int)
    return read_pyav(container, indices)

def get_video_features(video):
    inputs = processor(text=["placeholder"], videos=[video], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        return outputs.video_embeds  # shape [1, 512]

# --- Step 1: Precompute Video Features ---
feature_cache = {}

for row in tqdm(norm_df.iter_rows(named=True), total=norm_df.shape[0], desc="Precomputing features"):
    path = get_video_path(row["url"])
    if not path.exists():
        continue
    try:
        container = av.open(str(path))
        total = container.streams.video[0].frames
        if total < num_sampled_frames:
            continue
        vid = sample_frames(container, num_frames=num_sampled_frames)
        x = get_video_features(vid)
        feature_cache[row["url"]] = (x, row["target"])
    except Exception as e:
        print(f"Error processing {path.name}: {e}")

# --- Step 2: Collect Usable Rows ---
usable_rows = []
for row in tqdm(norm_df.iter_rows(named=True), total=norm_df.shape[0], desc="Collecting usable rows"):
    url = row["url"]
    if url not in feature_cache:
        continue
    video_emb, target = feature_cache[url]
    usable_rows.append((video_emb.squeeze(0), row["title"], target, url))

# --- Step 3: Batch Title Embeddings ---
video_embs, titles, targets, urls = zip(*usable_rows)

title_inputs = text_tokenizer(
    list(titles),
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=32
).to(device)

with torch.no_grad():
    title_outputs = text_model(**title_inputs)
    title_embs = title_outputs.last_hidden_state[:, 0, :]  # [N, 768]

# --- Step 4: Combine Video and Title Embeddings ---
video_embs_tensor = torch.stack([
    v.to(device) if isinstance(v, torch.Tensor) else torch.tensor(v).to(device)
    for v in video_embs
])  # [N, 512]

combined_embs = torch.cat([video_embs_tensor, title_embs], dim=-1)  # [N, 1280]
target_tensor = torch.tensor(targets, dtype=torch.float).unsqueeze(1).to(device)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

train_test_size = 0.1
batch_size = 4
# --- Step 5: Train/Test Split ---
X_train, X_test, y_train, y_test, urls_train, urls_test = train_test_split(
    combined_embs, target_tensor, list(urls), test_size=train_test_size, random_state=42
)

# --- Step 6: Dataloaders ---
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

In [ ]:
epochs = 64
learning_rate = 1e-5
dropout_rate = 0.5
validation_cadence = 1

# --- Model with Dropout ---
regressor = torch.nn.Sequential(
    torch.nn.Linear(1280, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(dropout_rate),
    torch.nn.Linear(256, 1)
).to(device)

optimizer = torch.optim.Adam(regressor.parameters(), lr=learning_rate)
loss_fn = torch.nn.MSELoss()  # Optional: replace with torch.nn.HuberLoss() if you expect outliers
# loss_fn = torch.nn.HuberLoss()  # Optional: replace with torch.nn.HuberLoss() if you expect outliers

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import tqdm

pbar = tqdm(total=epochs, desc="Training Progress", unit="epoch")

for epoch in range(epochs):
    pbar.update(1)
    regressor.train()
    
    total_loss = 0
    for xb, yb in train_loader:
        pred = regressor(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # --- Validation ---
    if (epoch + 1) % validation_cadence == 0:
        regressor.eval()
        with torch.no_grad():
            val_loss = 0
            for xb, yb in test_loader:
                pred = regressor(xb)
                loss = loss_fn(pred, yb)
                val_loss += loss.item()
            
            # Compute metrics
            train_preds = regressor(X_train).cpu().numpy()
            val_preds = regressor(X_test).cpu().numpy()
            y_train_cpu = y_train.cpu().numpy()
            y_test_cpu = y_test.cpu().numpy()

            train_mse = mean_squared_error(y_train_cpu, train_preds)
            train_r2 = r2_score(y_train_cpu, train_preds)
            val_mse = mean_squared_error(y_test_cpu, val_preds)
            val_r2 = r2_score(y_test_cpu, val_preds)

        print(f"Epoch {epoch + 1:>3} / {epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f}, "
              f"Val Loss: {val_loss / len(test_loader):.4f}, "
              f"Train MSE: {train_mse:.4f}, Train R2: {train_r2:.4f}, "
              f"Val MSE: {val_mse:.4f}, Val R2: {val_r2:.4f}")


In [ ]:
from transformers import AutoTokenizer, AutoModel

# Clone original dataframe
out_df = norm_df.clone()

# Map split assignments
split_map = {url: "train" for url in urls_train}
split_map.update({url: "test" for url in urls_test})

split_series = pl.Series(
    name="split",
    values=[split_map.get(url, None) for url in out_df["url"]]
)
out_df = out_df.with_columns(split_series)

# Reload tokenizer and model if needed
# text_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# text_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
# text_model.eval()
regressor.eval()

# Map url → prediction
url_to_pred = {}

with torch.no_grad():
    for row in tqdm(out_df.iter_rows(named=True), total=out_df.shape[0], desc="Predicting on test set"):
        url = row["url"]
        title = row["title"]

        if row["split"] != "test":
            continue

        # Must have video features
        if url not in feature_cache:
            continue

        video_emb, _ = feature_cache[url]
        video_emb = video_emb.squeeze(0)


        # Title embedding
        inputs = text_tokenizer(title, return_tensors="pt", truncation=True, padding=True, max_length=32).to(device)
        outputs = text_model(**inputs)
        title_emb = outputs.last_hidden_state[:, 0, :]  # [1, 768]
        title_emb = title_emb.squeeze(0)

        # Combine features and predict
        combined = torch.cat([video_emb, title_emb], dim=-1).unsqueeze(0)  # [1, 1280]
        pred = regressor(combined.to(device)).cpu().item()
        url_to_pred[url] = pred

# Add predictions to dataframe
pred_series = pl.Series(
    name="prediction",
    values=[url_to_pred.get(url, None) for url in out_df["url"]]
)

out_df = (
    out_df
    .with_columns(pred_series)
    .filter(pl.col("split") == "test")
    .with_columns(
        (pl.col("prediction") - pl.col("target")).abs().alias("prediction_error")
    )
)

# Evaluation
y_true = out_df["target"].to_numpy()
y_pred = out_df["prediction"].to_numpy()

mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test MSE: {mse:.4f}, R^2: {r2:.4f}")

# View sorted predictions
out_df.sort("prediction_error", descending=False).select(
    ["channel", "video_id", "title", "view_count", "target", "prediction", "prediction_error"]
)
